# RakForest — EDA: Forest Sound Dataset

Notebook นี้สำรวจ Dataset `forest_sound_dataset` ก่อนเทรนโมเดล โดยตรวจจำนวนไฟล์เสียง, ความยาว, sample rate, ไฟล์ที่อ่านไม่ได้ และสร้าง waveform / Mel-Spectrogram ตัวอย่างของแต่ละคลาส.


## 1. ติดตั้งไลบรารี (รันครั้งแรกครั้งเดียว)

หากยังไม่ได้ติดตั้ง ให้เอา `#` หน้าโค้ดออกแล้วรัน cell นี้.


In [ ]:
# %pip install pandas numpy librosa soundfile matplotlib seaborn


## 2. Import libraries และค้นหาโฟลเดอร์ Dataset


In [ ]:
from pathlib import Path
import warnings

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

AUDIO_EXTENSIONS = {'.wav', '.wave', '.mp3', '.flac', '.ogg', '.opus', '.m4a', '.aac'}
CLASS_NAMES = ['fire', 'logging', 'natural sound', 'poaching']

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent, cwd.parent.parent]
DATASET_DIR = next((root / 'forest_sound_dataset' for root in candidate_roots if (root / 'forest_sound_dataset').exists()), None)

if DATASET_DIR is None:
    raise FileNotFoundError(
        'ไม่พบ forest_sound_dataset — ให้วางโฟลเดอร์นี้ไว้ที่ root ของ repo PrePair_Sound แล้วรันใหม่'
    )

PROJECT_ROOT = DATASET_DIR.parent
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'reports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_DIR}')


## 3. สร้างตารางรายการไฟล์เสียง


In [ ]:
records = []
for label in CLASS_NAMES:
    class_dir = DATASET_DIR / label
    if not class_dir.exists():
        print(f'⚠️ ไม่พบโฟลเดอร์: {class_dir}')
        continue

    for path in class_dir.rglob('*'):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            records.append({'label': label, 'file_path': str(path), 'filename': path.name})

audio_df = pd.DataFrame(records)
if audio_df.empty:
    raise RuntimeError('ไม่พบไฟล์เสียงที่รองรับใน Dataset')

display(audio_df.head())
print(f'พบไฟล์เสียงทั้งหมด: {len(audio_df):,} ไฟล์')


## 4. จำนวนไฟล์ในแต่ละคลาส


In [ ]:
class_counts = audio_df['label'].value_counts().reindex(CLASS_NAMES, fill_value=0)
display(class_counts.rename('audio_files').to_frame())

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=class_counts.index, y=class_counts.values, hue=class_counts.index, legend=False, palette='viridis')
ax.set_title('จำนวนไฟล์เสียงในแต่ละคลาส')
ax.set_xlabel('คลาส')
ax.set_ylabel('จำนวนไฟล์')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 5. ตรวจความยาวเสียง, sample rate และไฟล์ที่อ่านไม่ได้


In [ ]:
def inspect_audio(path_str):
    try:
        info = sf.info(path_str)
        return pd.Series({
            'duration_seconds': round(info.frames / info.samplerate, 3),
            'sample_rate': info.samplerate,
            'channels': info.channels,
            'read_error': None,
        })
    except Exception as exc:
        return pd.Series({
            'duration_seconds': np.nan,
            'sample_rate': np.nan,
            'channels': np.nan,
            'read_error': str(exc),
        })

audio_details = audio_df['file_path'].apply(inspect_audio)
audio_df = pd.concat([audio_df, audio_details], axis=1)

broken_files = audio_df[audio_df['read_error'].notna()]
print(f'ไฟล์ที่อ่านไม่ได้: {len(broken_files):,} ไฟล์')
if not broken_files.empty:
    display(broken_files[['label', 'filename', 'read_error']].head(10))

summary = (audio_df.dropna(subset=['duration_seconds'])
           .groupby('label')
           .agg(audio_files=('filename', 'count'),
                mean_duration_s=('duration_seconds', 'mean'),
                min_duration_s=('duration_seconds', 'min'),
                max_duration_s=('duration_seconds', 'max'),
                sample_rates=('sample_rate', lambda x: sorted(x.dropna().unique().tolist())))
           .reindex(CLASS_NAMES))
display(summary)


## 6. ดูความยาวเสียงและ sample rate


In [ ]:
valid_df = audio_df.dropna(subset=['duration_seconds'])

plt.figure(figsize=(10, 5))
sns.boxplot(data=valid_df, x='label', y='duration_seconds', hue='label', legend=False, palette='Set2')
plt.title('การกระจายของความยาวไฟล์เสียง')
plt.xlabel('คลาส')
plt.ylabel('วินาที')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

display(pd.crosstab(valid_df['label'], valid_df['sample_rate']))


## 7. Waveform และ Mel-Spectrogram ตัวอย่าง


In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(16, 4 * len(CLASS_NAMES)))

for row_index, label in enumerate(CLASS_NAMES):
    sample_rows = valid_df[valid_df['label'] == label]
    if sample_rows.empty:
        axes[row_index, 0].set_axis_off()
        axes[row_index, 1].set_axis_off()
        continue

    sample = sample_rows.sample(1, random_state=42).iloc[0]
    y, sr = librosa.load(sample['file_path'], sr=None, mono=True)

    librosa.display.waveshow(y, sr=sr, ax=axes[row_index, 0])
    axes[row_index, 0].set_title(f'{label}: Waveform')

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    image = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[row_index, 1])
    axes[row_index, 1].set_title(f'{label}: Mel-Spectrogram')
    fig.colorbar(image, ax=axes[row_index, 1], format='%+2.0f dB')

plt.tight_layout()
plt.show()


## 8. บันทึกผลสรุป


In [ ]:
audio_df.to_csv(OUTPUT_DIR / 'forest_sound_audio_manifest.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'forest_sound_eda_summary.csv')

print('บันทึกไฟล์แล้ว:')
print(OUTPUT_DIR / 'forest_sound_audio_manifest.csv')
print(OUTPUT_DIR / 'forest_sound_eda_summary.csv')


## สิ่งที่ต้องสรุปหลังรัน

- จำนวนข้อมูลในแต่ละคลาสสมดุลหรือไม่
- ไฟล์เสียงมีความยาวใกล้เคียงกันหรือไม่
- sample rate เหมือนกันทั้งหมดหรือไม่
- มีไฟล์อ่านไม่ได้หรือไม่
- คลาสไหนควรเพิ่มข้อมูลหรือทำ data augmentation ในขั้นต่อไป
